# 14.14 怎樣讓多種終止輪數都能使用？


希望使用者可選一輪、三輪或五輪，就要讓訓練真的關心這些終止位置。資料仍然可以是同一問題和同一正解；要改的是模型在什麼深度交卷，以及哪裡被計分。

第一種配方是**抽終止深度**。每次拿到訓練題，從集合抽 R，展開 R 輪，僅在 `H(s_R)` 算交叉熵。若 R 抽自1、3、5，每個終止深度都會在部分更新中被直接監督。優化的是這些深度損失的加權平均；抽樣機率越高，它在長期期望目標裡的比重越高。

第二種配方是**逐輪監督／deep supervision**。一次展開到最大輪數，在多個中間狀態各算 loss，再加權成一個總 loss，例如 `(L_1+L_2+L_3)/3`。它用更多讀出與 loss 計算，鼓勵早期也能交卷。下面是兩種目標的寫法示意，數字是人工設定的 loss，不是實驗結果：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import random

loss_by_depth = {1: 1.2, 2: 0.8, 3: 0.7, 4: 0.9, 5: 0.6}
rng = random.Random(42)
R = rng.choice([1, 3, 5])
sampled_loss = loss_by_depth[R]
deep_loss = sum(loss_by_depth[r] for r in range(1, 6)) / 5
print("本次終止", R, "末端loss", sampled_loss, "逐輪平均", deep_loss)

在真正的 PyTorch 訓練裡，loss 必須是保留計算圖的 tensor，不能像這段展示程式換成 Python 浮點數後再反傳。這裡只讓讀者核對「哪些項進入目標」；正式實作與配方會另存。

同一正解不奇怪：老師要求不同時間交卷都答同一題，並沒有要求內部狀態完全相同。早期輪數可能只能處理簡單題，後期可再處理複雜題；這是值得測試的學習結果，並不是 loss 寫下去就會出現。多個深度也會競爭共享參數，過度偏重一輪可能犧牲另一輪。

輪數多也會增加每次更新的工作，不能只把「更新步數相同」當成相同訓練成本。若做效果比較，記抽樣分布、平均展開輪數、監督位置、有效題數與實際時間；若只教機制，短配方足夠，不宣稱已重現研究規模效益。

練習：抽到5輪後，又在第2與4輪加 CE，還能叫「只在奇數終止深度監督」嗎？不能。這正是下一節要守住的實驗界線。
